# Inspect the pretrained Seeker weights

Before training anything, load the released `seeker.mimicgen.pth` checkpoint and look at what it actually predicts: a soft attention mask over the DINOv3 patch grid, and the tight bounding box derived from it, for both the agentview and eye-in-hand cameras.

This is a sanity check, not a benchmark — it uses whichever cached task you point it at and a handful of frames from one demo.

**Prerequisites**

- `seeker` conda environment created and activated (see the top-level [README](../README.md#2-create-the-environment))
- `seeker setup` has been run (downloads `.weights/seeker.mimicgen.pth` and `.weights/dinov3.vits16plus.pth`)
- At least one task has been rerendered to the LMDB cache format, e.g.:

  ```bash
  seeker rerender-dataset --dataset datasets/mimicgen/three_piece_assembly_d2/three_piece_assembly_d2.hdf5 --n-demo 100
  ```

The first code cell hops up to the repository root if needed, so this works whether you launch Jupyter from the repo root or open the notebook directly from `notebooks/`.

In [ ]:
import os

import lmdb
import numpy as np
import torch
from IPython.display import Image as IPyImage, display

# Jupyter kernels start with cwd = this notebook's own directory, not the repo
# root, regardless of where `jupyter notebook`/`jupyter lab` was launched from.
# Hop up so relative paths (`.weights/...`, `datasets/...`) resolve.
if not os.path.isfile("setup.py") and os.path.isfile("../setup.py"):
    os.chdir("..")

from seeker.dataset.cache import load_lowdim, load_metadata, load_numpy_array, resolve_cache_dir
from seeker.model.base_encoder import BaseEncoder
from seeker.model.seeker import Seeker
from seeker.util.image_io import decode_jpg_bytes
from seeker.util.roi import grid_mask_to_pixel_box
from seeker.util.visualization import visualize

## Configuration

Point `DATASET_PATH` at any task you've already run `seeker rerender-dataset` on.

In [ ]:
DATASET_PATH = "datasets/mimicgen/three_piece_assembly_d2/three_piece_assembly_d2.hdf5"
WEIGHTS_PATH = ".weights/seeker.mimicgen.pth"

VIT_IN = 224  # Seeker's input resolution (patch_size=16 -> 14x14 patch grid)
EPISODE_IDX = 0
N_FRAMES = 4  # frames sampled evenly across the episode

CACHE_DIR = resolve_cache_dir(DATASET_PATH)
print("cache dir:", CACHE_DIR)

## Load a handful of real frames from the cache

Reads directly from the rerendered LMDB cache (the same cache `seeker train` and `seeker playback-dataset` read from), without going through the full training `Dataset`/`DataLoader` stack.

In [ ]:
def load_obs_window(cache_dir, *, episode_idx: int, n_frames: int, vit_in: int):
    """Decode `n_frames` evenly spaced frames of one episode into a model-ready obs dict."""
    meta, episode_lengths = load_metadata(cache_dir)
    rgb_keys = meta["rgb_keys"]
    lowdim_keys = meta["lowdim_keys"]
    assert {"agentview_image", "eye_in_hand_image"}.issubset(rgb_keys), rgb_keys

    ep_len = episode_lengths[episode_idx]
    n = min(n_frames, ep_len)
    local_idx = np.linspace(0, ep_len - 1, num=n, dtype=np.int64)
    episode_start = int(sum(episode_lengths[:episode_idx]))
    global_idx = episode_start + local_idx

    lowdim = load_lowdim(cache_dir, lowdim_keys)

    frames = {}
    env = lmdb.open(os.path.join(str(cache_dir), "images.lmdb"), subdir=False, readonly=True, lock=False)
    with env.begin() as txn:
        for img_key in ("agentview_image", "eye_in_hand_image"):
            imgs = []
            for gidx in global_idx.tolist():
                buf = txn.get(f"{img_key}/{int(gidx):08d}".encode("ascii"))
                if buf is None:
                    raise KeyError(f"Missing LMDB key: {img_key}/{gidx:08d}")
                imgs.append(decode_jpg_bytes(buf, image_size=vit_in, to_float=True, fmt="CHW"))
            frames[img_key] = np.stack(imgs, axis=0)  # [n, 3, vit_in, vit_in]
    env.close()

    task_embedding = load_numpy_array(cache_dir, "lowdim/task_embedding.npy")[episode_idx]
    robot_id = int(load_numpy_array(cache_dir, "lowdim/robot_id.npy")[episode_idx])

    # [N, T=1, ...] -- BaseEncoder.obs_to_input expects a temporal dimension.
    obs = {
        "agentview_image": torch.from_numpy(frames["agentview_image"]).unsqueeze(1).float(),
        "eye_in_hand_image": torch.from_numpy(frames["eye_in_hand_image"]).unsqueeze(1).float(),
        "robot0_eef_pos": torch.from_numpy(lowdim["robot0_eef_pos"][global_idx].copy()).unsqueeze(1),
        "robot0_eef_rot": torch.from_numpy(lowdim["robot0_eef_rot"][global_idx].copy()).unsqueeze(1),
        "robot0_gripper_qpos": torch.from_numpy(lowdim["robot0_gripper_qpos"][global_idx].copy()).unsqueeze(1),
        "robot_id": torch.full((n, 1, 1), float(robot_id)),
        "task_embedding": torch.from_numpy(task_embedding.copy()).unsqueeze(0).unsqueeze(0).expand(n, 1, -1),
    }
    return obs


obs = load_obs_window(CACHE_DIR, episode_idx=EPISODE_IDX, n_frames=N_FRAMES, vit_in=VIT_IN)
{k: v.shape for k, v in obs.items()}

## Load the pretrained Seeker checkpoint

`strict_weights=True` means this will raise immediately if the checkpoint doesn't match the model config — see [`seeker/model/WEIGHTS.md`](../seeker/model/WEIGHTS.md) for the compatibility contract. The checkpoint also carries the normalizer stats it was trained with, so no separate normalizer needs to be built here.

In [ ]:
seeker = Seeker(
    weights=WEIGHTS_PATH,
    views=["agentview", "eye_in_hand"],
    verbose=True,
    strict_weights=True,
)
seeker.eval();

## Run Seeker on the frames

`BaseEncoder.obs_to_input` does the same normalization Seeker was trained with (ImageNet image stats, per-robot proprio/task-embedding normalization from the checkpoint). Each view then runs through Seeker's coarse stage, then its fine stage (restricted to the coarse mask's region) — the same two-stage refinement used everywhere else in the repo.

In [ ]:
encoder = BaseEncoder()
encoder.enable_eih = True
enc_in = encoder.obs_to_input(obs, seeker.normalizer, resize=False)

n = enc_in.agentview.shape[0]
default_box = torch.tensor([[0.0, 0.0, VIT_IN - 1.0, VIT_IN - 1.0]]).expand(n, -1)

with torch.no_grad():
    agent_out = seeker(image=enc_in.agentview, view="agentview", composer_in=enc_in.composer_in, stage="fine")
    eih_out = seeker(image=enc_in.eye_in_hand, view="eye_in_hand", composer_in=enc_in.composer_in, stage="fine")

agent_coarse_box = grid_mask_to_pixel_box(agent_out.coarse.mask.squeeze(1), default_box)
agent_fine_box = grid_mask_to_pixel_box(agent_out.final.mask.squeeze(1), default_box)
eih_coarse_box = grid_mask_to_pixel_box(eih_out.coarse.mask.squeeze(1), default_box)
eih_fine_box = grid_mask_to_pixel_box(eih_out.final.mask.squeeze(1), default_box)

One row per frame; columns are agentview coarse mask, agentview fine mask, eye-in-hand coarse mask, eye-in-hand fine mask. The red box is the tight crop `grid_mask_to_pixel_box` derives from each mask — this is exactly what `method=seeker` policy training crops to (see the README's [Training](../README.md#training)).

In [ ]:
views = [
    (enc_in.agentview, agent_out.coarse.mask, agent_coarse_box, None),
    (enc_in.agentview, agent_out.final.mask, agent_fine_box, None),
    (enc_in.eye_in_hand, eih_out.coarse.mask, eih_coarse_box, None),
    (enc_in.eye_in_hand, eih_out.final.mask, eih_fine_box, None),
]

viz_dir = "notebooks/_viz"
visualize(views, temporal_dim=1, num_viz=n, save_dir=viz_dir, step=0)
display(IPyImage(filename=os.path.join(viz_dir, "steps_000000_009999", "train_step_0_viz.png")))